# Exercise 3: Purchase Order Price Anomaly Detection

In this notebook, we consume the **Purchase Order** data product ([`Purchase Order`](https://api.sap.com/dataproduct/sap-s4com-PurchaseOrder-v1/overview)) from the SAP BDC Catalog. It is an SAP-managed data product from SAP S/4HANA Private Cloud Edition shared here through the SAP BDC Connect component.

We use **statistical anomaly detection** (composite z-score + PERCENT_RANK) entirely in SQL to identify purchase order line items with unusual pricing, then publish the enriched results back to the BDC Catalog as a custom data product. This ML technique is valid but on the simpler end of the spectrum to keep the use case simple. 

> [!NOTE]
> For a more advanced scenario for expert data scientists, you can consider Snowflake's built-in SNOWFLAKE.ML.ANOMALY_DETECTION

> [!IMPORTANT]
> <b>In the notebook, please pay attention to the ✍️ icon. Whenever there is an instruction with the ✍️ icon, edit the code to fit your own user. </b>

## Step 0 — Set Context and create schema
✍️ Replace the `<SCHEMANAME>` with your username. Eg. CREATE SCHEMA IF NOT ESISTS TECHED_DA162.SCHEMA_AC1262739283

In [ ]:
%%sql -r ctx
--STOP: before running, change the SCHEMA NAME to your USERNAME
USE WAREHOUSE SYSTEM$STREAMLIT_NOTEBOOK_WH;
USE DATABASE TECHED_DA162;
CREATE SCHEMA IF NOT EXISTS TECHED_DA162.<SCHEMANAME>


## Step 1 — Read the SAP Managed Data Product

The **Purchase Order** data product ([`sap-s4com-PurchaseOrder-v1`](https://api.sap.com/dataproduct/sap-s4com-PurchaseOrder-v1/overview)) is available via the BDC Catalog share.
We select the columns relevant for price anomaly detection from the `purchaseorderitem` entity and filter to valid records.

Note that the table is the data product table that has been mounted. PURCHASEORDER_V1."purchaseorder"."purchaseorderitem" (Database.schema.table)

In [ ]:
%%sql -r preview_data
SELECT
    PurchaseOrder                   AS PURCHASING_DOCUMENT,
    PurchaseOrderItem               AS PURCHASING_DOCUMENT_ITEM,
    Material                        AS MATERIAL,
    MaterialGroup                   AS MATERIAL_GROUP,
    Supplier                        AS SUPPLIER,
    Plant                           AS PLANT,
    CompanyCode                     AS COMPANY_CODE,
    PurchaseOrderDate               AS DOCUMENT_DATE,
    NetPriceAmount                  AS NET_PRICE,
    OrderQuantity                   AS ORDER_QUANTITY,
    ROUND(NetPriceAmount * OrderQuantity, 2) AS NET_ORDER_VALUE
FROM PURCHASEORDER_V1."purchaseorder"."purchaseorderitem"
WHERE NetPriceAmount > 0
  AND OrderQuantity > 0
LIMIT 20;

In [ ]:
%%sql -r row_count
SELECT COUNT(*) AS total_records
FROM PURCHASEORDER_V1."purchaseorder"."purchaseorderitem"
WHERE NetPriceAmount > 0
  AND OrderQuantity > 0;

## Step 2 — Feature Engineering

Compute per-material price statistics using window functions:
- **AVG_PRICE_BY_MATERIAL** — average price for each material
- **STDDEV_PRICE_BY_MATERIAL** — standard deviation of price per material
- **PRICE_DEVIATION** — z-score: how many std devs this price is from the material mean
- **PRICE_RATIO** — ratio of this price to the material average


✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and the `<TABLENAME>` with `po_features`

In [ ]:
%%sql -r features_created
----STOP: before running, change the SCHEMA NAME to your USERNAME and TABLENAME to an appropriate name
CREATE OR REPLACE TEMPORARY TABLE TECHED_DA162.<SCHEMANAME>.<TABLENAME> AS
SELECT
    PurchaseOrder                   AS PURCHASING_DOCUMENT,
    PurchaseOrderItem               AS PURCHASING_DOCUMENT_ITEM,
    Material                        AS MATERIAL,
    MaterialGroup                   AS MATERIAL_GROUP,
    Supplier                        AS SUPPLIER,
    Plant                           AS PLANT,
    CompanyCode                     AS COMPANY_CODE,
    PurchaseOrderDate               AS DOCUMENT_DATE,
    NetPriceAmount                  AS NET_PRICE,
    OrderQuantity                   AS ORDER_QUANTITY,
    ROUND(NetPriceAmount * OrderQuantity, 2) AS NET_ORDER_VALUE,

    -- Per-material price statistics
    AVG(NetPriceAmount)    OVER (PARTITION BY Material) AS AVG_PRICE_BY_MATERIAL,
    STDDEV(NetPriceAmount) OVER (PARTITION BY Material) AS STDDEV_PRICE_BY_MATERIAL,

    -- Z-score: how far is this price from the material average?
    COALESCE(
        (NetPriceAmount - AVG(NetPriceAmount) OVER (PARTITION BY Material))
        / NULLIF(STDDEV(NetPriceAmount) OVER (PARTITION BY Material), 0),
        0
    ) AS PRICE_DEVIATION,

    -- Ratio of price to material average
    NetPriceAmount / NULLIF(AVG(NetPriceAmount) OVER (PARTITION BY Material), 0) AS PRICE_RATIO

FROM PURCHASEORDER_V1."purchaseorder"."purchaseorderitem"
WHERE NetPriceAmount > 0
  AND OrderQuantity > 0;

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and the `<TABLENAME>` with `po_features`

In [ ]:
%%sql -r price_stats
SELECT
    COUNT(*)                        AS total_rows,
    AVG(NET_PRICE)                  AS avg_price,
    STDDEV(NET_PRICE)               AS stddev_price,
    MEDIAN(NET_PRICE)               AS median_price,
    MIN(NET_PRICE)                  AS min_price,
    MAX(NET_PRICE)                  AS max_price,
    AVG(ORDER_QUANTITY)             AS avg_quantity,
    AVG(ABS(PRICE_DEVIATION))       AS avg_abs_zscore
FROM TECHED_DA162.<SCHEMANAME>.<TABLENAME>;

## Step 3 — Anomaly Detection

We use a **composite anomaly score** combining multiple z-scores, then apply `PERCENT_RANK` to flag the top 5% as anomalies — matching a 5% contamination rate.

- **ANOMALY_FLAG** = `YES` / `NO`
- **ANOMALY_DIRECTION** = `OVERPRICED` / `UNDERPRICED` / `NORMAL`
- **ANOMALY_CONFIDENCE** = 0–100 based on z-score magnitude

This is a purely statistical approach that runs entirely in SQL.

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and the `<TABLENAME>` with po_scored

In [ ]:
%%sql -r scored_created
CREATE OR REPLACE TABLE TECHED_DA162.<SCHEMANAME>.<TABLENAME> AS --CHANGE the Schema and table name
WITH feature_zscores AS (
    SELECT
        *,
        -- Z-scores for each feature (how far from overall mean in std devs)
        COALESCE((NET_PRICE       - AVG(NET_PRICE)       OVER ()) / NULLIF(STDDEV(NET_PRICE)       OVER (), 0), 0) AS Z_NET_PRICE,
        COALESCE((ORDER_QUANTITY   - AVG(ORDER_QUANTITY)  OVER ()) / NULLIF(STDDEV(ORDER_QUANTITY)  OVER (), 0), 0) AS Z_ORDER_QUANTITY,
        COALESCE((NET_ORDER_VALUE  - AVG(NET_ORDER_VALUE) OVER ()) / NULLIF(STDDEV(NET_ORDER_VALUE) OVER (), 0), 0) AS Z_NET_ORDER_VALUE,
        COALESCE(PRICE_RATIO - 1, 0)                                                                                AS PRICE_RATIO_DEVIATION
    FROM TECHED_DA162.<SCHEMANAME>.po_features --CHANGE the Schema name
),
scored AS (
    SELECT
        *,
        -- Composite anomaly score: Euclidean distance in z-score space
        SQRT(
            POWER(PRICE_DEVIATION, 2) +
            POWER(Z_NET_PRICE, 2) +
            POWER(Z_ORDER_QUANTITY, 2) +
            POWER(Z_NET_ORDER_VALUE, 2) +
            POWER(PRICE_RATIO_DEVIATION, 2)
        ) AS ANOMALY_SCORE
    FROM feature_zscores
)
SELECT
    *,
    -- Flag top 5% by anomaly score as anomalies
    CASE
        WHEN PERCENT_RANK() OVER (ORDER BY ANOMALY_SCORE DESC) <= 0.05 THEN 'YES'
        ELSE 'NO'
    END AS ANOMALY_FLAG,

    -- Direction: overpriced or underpriced?
    CASE
        WHEN PERCENT_RANK() OVER (ORDER BY ANOMALY_SCORE DESC) <= 0.05
             AND PRICE_DEVIATION > 0 THEN 'OVERPRICED'
        WHEN PERCENT_RANK() OVER (ORDER BY ANOMALY_SCORE DESC) <= 0.05
             AND PRICE_DEVIATION <= 0 THEN 'UNDERPRICED'
        ELSE 'NORMAL'
    END AS ANOMALY_DIRECTION,

    -- Confidence score based on price deviation magnitude (0-100)
    CASE
        WHEN ABS(PRICE_DEVIATION) > 4 THEN 100
        WHEN ABS(PRICE_DEVIATION) > 3 THEN 85
        WHEN ABS(PRICE_DEVIATION) > 2 THEN 70
        WHEN ABS(PRICE_DEVIATION) > 1 THEN 55
        ELSE 30
    END AS ANOMALY_CONFIDENCE
FROM scored;

## Step 4 — Score & Flag Anomalies

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username

In [ ]:
%%sql -r sample_anomalies
SELECT
    PURCHASING_DOCUMENT,
    MATERIAL,
    SUPPLIER,
    NET_PRICE,
    ROUND(AVG_PRICE_BY_MATERIAL, 2)  AS AVG_PRICE_BY_MATERIAL,
    ROUND(PRICE_DEVIATION, 2)        AS PRICE_DEVIATION,
    ROUND(ANOMALY_SCORE, 2)          AS ANOMALY_SCORE,
    ANOMALY_FLAG,
    ANOMALY_DIRECTION,
    ANOMALY_CONFIDENCE
FROM TECHED_DA162.<SCHEMANAME>.po_scored --CHANGE the schema name to include your username
WHERE ANOMALY_FLAG = 'YES'
ORDER BY ANOMALY_SCORE DESC
LIMIT 15;

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username

In [ ]:
%%sql -r summary_stats
SELECT
    COUNT(*)                                                     AS total_po_line_items,
    SUM(CASE WHEN ANOMALY_FLAG = 'YES' THEN 1 ELSE 0 END)       AS anomalies_detected,
    ROUND(SUM(CASE WHEN ANOMALY_FLAG = 'YES' THEN 1 ELSE 0 END)
          / COUNT(*) * 100, 1)                                   AS anomaly_rate_pct,
    ROUND(AVG(CASE WHEN ANOMALY_FLAG = 'YES' THEN ANOMALY_SCORE END), 2) AS avg_anomaly_score,
    ROUND(AVG(CASE WHEN ANOMALY_FLAG = 'NO'  THEN ANOMALY_SCORE END), 2) AS avg_normal_score
FROM TECHED_DA162.<SCHEMANAME>.po_scored; --CHANGE the schema name to include your username

## Step 5 — Create the Custom Data table

Write the scored results as a persistent table for consumption by other teams.

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and `<TABLENAME>` with `PO_PRICE_ANOMALY_SCORES`

In [ ]:
%%sql -r dp_created
CREATE OR REPLACE ICEBERG TABLE TECHED_DA162.<SCHEMANAME>.<TABLENAME>
  ICEBERG_VERSION = 3
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
  ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
  STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
  AS
SELECT
    -- Original PO fields
    PURCHASING_DOCUMENT,
    PURCHASING_DOCUMENT_ITEM,
    MATERIAL,
    MATERIAL_GROUP,
    SUPPLIER,
    PLANT,
    COMPANY_CODE,
    DOCUMENT_DATE,
    NET_PRICE,
    ORDER_QUANTITY,
    NET_ORDER_VALUE,

    -- ML enrichment
    ROUND(AVG_PRICE_BY_MATERIAL, 2)  AS AVG_PRICE_BY_MATERIAL,
    ROUND(PRICE_DEVIATION, 4)        AS PRICE_DEVIATION,
    ROUND(PRICE_RATIO, 4)            AS PRICE_RATIO,
    ROUND(ANOMALY_SCORE, 4)          AS ANOMALY_SCORE,
    ANOMALY_FLAG,
    ANOMALY_DIRECTION,
    ANOMALY_CONFIDENCE,

    -- Metadata
    CURRENT_TIMESTAMP()              AS SCORED_AT

FROM TECHED_DA162.<SCHEMANAME>.po_scored;

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and `<TABLENAME>` with `PO_PRICE_ANOMALY_SCORES`

In [ ]:
%%sql -r verify_output
SELECT * FROM TECHED_DA162.<SCHEMANAME>.<TABLENAME>
LIMIT 10;

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and `<TABLENAME>` with `PO_PRICE_ANOMALY_SCORES`

In [ ]:
%%sql -r flag_breakdown
SELECT
    ANOMALY_FLAG,
    ANOMALY_DIRECTION,
    COUNT(*)                                AS record_count,
    ROUND(AVG(NET_PRICE), 2)                AS avg_net_price,
    ROUND(AVG(ABS(PRICE_DEVIATION)), 2)     AS avg_abs_deviation,
    ROUND(AVG(ANOMALY_CONFIDENCE), 0)       AS avg_confidence
FROM TECHED_DA162.<SCHEMANAME>.<TABLENAME>
GROUP BY ANOMALY_FLAG, ANOMALY_DIRECTION
ORDER BY ANOMALY_FLAG DESC, ANOMALY_DIRECTION;

## [OPTIONAL] Anomaly Analysis — Conclusions 

✍️ Replace the `<SCHEMANAME>` with your SCHEMA_username and `<TABLENAME>` with `PO_PRICE_ANOMALY_SCORES`

In [ ]:
%%sql -r anomaly_by_group
SELECT
    MATERIAL_GROUP,
    COUNT(*) AS total_items,
    SUM(CASE WHEN ANOMALY_FLAG = 'YES' THEN 1 ELSE 0 END) AS anomaly_count,
    ROUND(SUM(CASE WHEN ANOMALY_FLAG = 'YES' THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS anomaly_rate_pct,
    ROUND(SUM(CASE WHEN ANOMALY_FLAG = 'YES' THEN NET_ORDER_VALUE ELSE 0 END), 0) AS value_at_risk
FROM TECHED_DA162.<SCHEMANAME>.<TABLENAME>
GROUP BY MATERIAL_GROUP
HAVING COUNT(*) >= 5
ORDER BY anomaly_rate_pct DESC;

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = anomaly_by_group.to_pandas()

# Top 12 material groups by anomaly rate (with enough volume)
df_top = df[df['TOTAL_ITEMS'] >= 10].head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Chart 1: Anomaly Rate by Material Group ---
colors = ['#BB0000' if r >= 20 else '#E78C07' if r >= 5 else '#0070F2'
          for r in df_top['ANOMALY_RATE_PCT']]
axes[0].barh(df_top['MATERIAL_GROUP'][::-1], df_top['ANOMALY_RATE_PCT'][::-1], color=colors[::-1])
axes[0].set_xlabel('Anomaly Rate (%)')
axes[0].set_title('Price Anomaly Rate by Material Group')
axes[0].axvline(x=5, color='gray', linestyle='--', alpha=0.5, label='5% baseline')
for i, (rate, cnt) in enumerate(zip(df_top['ANOMALY_RATE_PCT'][::-1], df_top['ANOMALY_COUNT'][::-1])):
    axes[0].text(rate + 0.5, i, f'{rate}% ({cnt:,})', va='center', fontsize=8)
axes[0].legend()

# --- Chart 2: Value at Risk by Material Group ---
df_risk = df.nlargest(10, 'VALUE_AT_RISK')
colors2 = ['#BB0000' if r >= 20 else '#E78C07' if r >= 5 else '#0070F2'
           for r in df_risk['ANOMALY_RATE_PCT']]
axes[1].barh(df_risk['MATERIAL_GROUP'][::-1],
             df_risk['VALUE_AT_RISK'][::-1] / 1e6, color=colors2[::-1])
axes[1].set_xlabel('Value at Risk (M)')
axes[1].set_title('Anomalous Order Value by Material Group')
for i, (val, rate) in enumerate(zip(df_risk['VALUE_AT_RISK'][::-1], df_risk['ANOMALY_RATE_PCT'][::-1])):
    axes[1].text(val / 1e6 + 0.01, i, f'{val/1e6:,.1f}M ({rate}%)', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Chart: Anomalous vs Normal — Price & Quantity Profile ---
from snowflake.snowpark.context import get_active_session
session = get_active_session()

df_all = session.table('PO_PRICE_ANOMALY_SCORES').to_pandas()

df_normal  = df_all[df_all['ANOMALY_FLAG'] == 'NO']
df_anomaly = df_all[df_all['ANOMALY_FLAG'] == 'YES']

normal_avg_price   = df_normal['NET_PRICE'].mean()
anomaly_avg_price  = df_anomaly['NET_PRICE'].mean()
normal_avg_qty     = df_normal['ORDER_QUANTITY'].mean()
anomaly_avg_qty    = df_anomaly['ORDER_QUANTITY'].mean()
normal_avg_zscore  = df_normal['PRICE_DEVIATION'].abs().mean()
anomaly_avg_zscore = df_anomaly['PRICE_DEVIATION'].abs().mean()

normal_count  = len(df_normal)
anomaly_count = len(df_anomaly)

df_summary = pd.DataFrame({
    'Metric': ['Avg Net Price', 'Avg Order Qty', 'Avg |Z-Score|'],
    'Normal':  [normal_avg_price, normal_avg_qty, normal_avg_zscore],
    'Anomaly': [anomaly_avg_price, anomaly_avg_qty, anomaly_avg_zscore]
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, (_, row) in enumerate(df_summary.iterrows()):
    bars = axes[i].bar(
        [f'Normal\n({normal_count:,} items)', f'Anomaly\n({anomaly_count:,} items)'],
        [row['Normal'], row['Anomaly']],
        color=['#0070F2', '#BB0000'], width=0.5
    )
    axes[i].set_title(row['Metric'], fontsize=13, fontweight='bold')
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                     f'{bar.get_height():,.2f}', ha='center', va='bottom', fontsize=10)
    axes[i].set_ylabel('')

price_ratio = anomaly_avg_price / normal_avg_price if normal_avg_price > 0 else 0
qty_ratio   = anomaly_avg_qty / normal_avg_qty if normal_avg_qty > 0 else 0

fig.suptitle(
    f'Anomalous POs: {price_ratio:.0f}x Higher Prices, {qty_ratio:.0f}x Larger Quantities',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

print(f'\n--- Key Findings ---')
print(f'  Total PO line items: {len(df_all):,}')
print(f'  Normal: {normal_count:,} | Anomalous: {anomaly_count:,}')
print(f'  Anomaly rate: {anomaly_count / len(df_all) * 100:.1f}%')
print(f'  Avg anomalous price: {anomaly_avg_price:,.2f} vs normal: {normal_avg_price:,.2f} ({price_ratio:.0f}x)')
print(f'  Avg anomalous quantity: {anomaly_avg_qty:,.0f} vs normal: {normal_avg_qty:,.0f} ({qty_ratio:.0f}x)')